# Bissett 2019 Figure Recreations — experiment_3

Recreates Figures 1a-b, 2a-d, and 3a from Bissett, Jones, Poldrack & Logan (2020), *Severe and pervasive violations of independence in response inhibition tasks*, using the experiment_3 dataset (Simple Stop + Working Memory Task memory loads).

The original paper analyses 25 between-subjects conditions; here each "condition" is one block-type-or-load within a single dataset (Simple Stop, plus each memory-load level). The paper's LME-derived 95% CI on the group mean is replaced with a subject-level bootstrap CI to keep this notebook pure-Python.

## Table of Contents

1. Imports and data loading
2. Helper functions (bootstrap CI; condition assembly)
3. Figure 1a-b — per-condition violation curves
4. Figure 2a — violations by SSD across all conditions
5. Figure 2b — proportion of conditions that violate at each SSD
6. Figure 2c — proportion of subjects who violate at each SSD
7. Figure 2d — cumulative stop-trial distribution by SSD
8. Figure 3a — SSRT (all SSDs) vs. SSRT (SSD >= 200 ms)

Faithful to:
`violations_paper_codebase/henrymj/henrymj-ContextDependence-a1c0d79/generate_figures.ipynb`.


## 1. Imports and data loading

In [ ]:
#imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from stop_wm.config import ProjectConfig
from stop_wm.race_model_check import (
    compute_per_subject_per_ssd_preceding,
    ssrt_short_vs_long_comparison,
)
from stop_wm.subjectwise_metrics import calculate_ssrt_integration

project_root = Path.cwd().parent
config = ProjectConfig(project_root=project_root)

# Pathing
trial_wise_data_wm_path = config.results_dir / 'post_qc_stop_signal_wm_trials.csv'
trial_wise_data_stop_path = config.results_dir / 'post_qc_stop_signal_trials.csv'

# Load the data
trial_wise_data_wm = pd.read_csv(trial_wise_data_wm_path)
trial_wise_data_stop = pd.read_csv(trial_wise_data_stop_path)


## 2. Helper functions

In [ ]:
# === Helpers: bootstrap CI for the group mean by SSD; condition assembly ===
SIMPLE_COLOR = '#C1121F'
LOAD_PALETTE = ['#2E86AB', '#A23B72', '#F18F01', '#6A4C93']

def bootstrap_mean_ci_by_ssd(per_subject, ssd_col, value_col='mean_violation_ms',
                             n_boot=2000, alpha=0.05, min_subjects=5, seed=0):
    """Bootstrap 95% CI on the group mean of `value_col` per SSD.

    Resamples subjects with replacement at each SSD. SSDs with fewer than
    `min_subjects` are dropped (matches Bissett's >=5 subs rule).
    Returns DataFrame: ssd, n_subjects, mean, lo, hi.
    """
    rng = np.random.default_rng(seed)
    rows = []
    for ssd, grp in per_subject.groupby(ssd_col):
        vals = grp[value_col].dropna().values
        n = len(vals)
        if n < min_subjects:
            continue
        boot_means = np.empty(n_boot)
        for b in range(n_boot):
            sample = rng.choice(vals, size=n, replace=True)
            boot_means[b] = sample.mean()
        lo, hi = np.quantile(boot_means, [alpha/2, 1 - alpha/2])
        rows.append({ssd_col: ssd, 'n_subjects': n,
                     'mean': float(vals.mean()), 'lo': float(lo), 'hi': float(hi)})
    return pd.DataFrame(rows).sort_values(ssd_col).reset_index(drop=True)


def condition_palette(loads):
    """Return a list of {label, color, df-getter} dicts: Simple Stop + each load."""
    out = [{'label': 'Simple Stop', 'color': SIMPLE_COLOR, 'tag': 'simple', 'load': None}]
    for i, L in enumerate(loads):
        out.append({
            'label': f'Load {int(L)}',
            'color': LOAD_PALETTE[i % len(LOAD_PALETTE)],
            'tag': f'load_{int(L)}',
            'load': float(L),
        })
    return out


# Compute the paired-preceding diagnostic once for each dataset; we'll reuse
# it across most figures.
race_simple = compute_per_subject_per_ssd_preceding(
    trial_wise_data_stop,
    trial_type_col='trial_SS_trial_type', rt_col='trial_rt',
    correct_col='trial_correct_trial', ssd_col='trial_SSD',
    subject_col='participant_id', block_col='block_num',
    trial_order_col='current_trial',
)
race_dual = compute_per_subject_per_ssd_preceding(
    trial_wise_data_wm,
    trial_type_col='stop_trial_SS_trial_type', rt_col='stop_trial_rt',
    correct_col='stop_trial_correct_trial', ssd_col='stop_trial_SSD',
    subject_col='participant_id', block_col='block_num',
    trial_order_col='current_trial',
    condition_col='memory_trial_stimLength',
)

LOADS = sorted(race_dual['memory_trial_stimLength'].dropna().unique())
CONDITIONS = condition_palette(LOADS)
print('Conditions:', [c['label'] for c in CONDITIONS])
print('Simple-stop subjects:', race_simple['participant_id'].nunique())
for L in LOADS:
    n = race_dual[race_dual['memory_trial_stimLength'] == L]['participant_id'].nunique()
    print(f'  Load {int(L)}: {n} subjects with at least one valid (subject, SSD) cell')


## 3. Figure 1a-b — per-condition violation curves

The paper's Figure 1 plots violation = stop-failure RT − preceding go RT against SSD, with one panel per *condition*: every subject's curve (thin, colored) plus a black group-mean line with 95% CI. The original shows just FixedSSDs1 and FixedSSDs2; here we extend to one panel per condition in this dataset (Simple Stop + each memory load).

Bootstrap 95% CIs over subjects replace the paper's R/lme4 mixed-model CI.


In [ ]:
# === Figure 1a-b style: per-condition curves with subject lines + group mean+CI ===
n_panels = len(CONDITIONS)
fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 6), sharey=True)
if n_panels == 1:
    axes = [axes]

for ax, cond in zip(axes, CONDITIONS):
    if cond['load'] is None:
        per = race_simple.copy()
        ssd_col = 'trial_SSD'
    else:
        per = race_dual[race_dual['memory_trial_stimLength'] == cond['load']].copy()
        ssd_col = 'stop_trial_SSD'

    # Per-subject pivot: rows = SSD, columns = subject
    pivot = per.pivot_table(values='mean_violation_ms',
                            index=ssd_col, columns='participant_id')
    # Plot each subject as a thin colored line
    for col in pivot.columns:
        ax.plot(pivot.index, pivot[col].values, color=cond['color'], alpha=0.35, linewidth=1.0)

    # Bootstrap group mean + 95% CI
    boot = bootstrap_mean_ci_by_ssd(per, ssd_col=ssd_col, value_col='mean_violation_ms')
    if not boot.empty:
        ax.plot(boot[ssd_col], boot['mean'], color='black', linewidth=3.5, zorder=5)
        ax.fill_between(boot[ssd_col], boot['lo'], boot['hi'],
                        color='black', alpha=0.18, zorder=4)

    ax.axhline(0, color='black', linestyle=':', linewidth=1.2)
    ax.set_xlabel('Stop-signal delay (ms)', fontsize=13, fontweight='bold')
    ax.set_title(cond['label'], fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

axes[0].set_ylabel('Stop-failure RT − preceding go RT (ms)',
                   fontsize=13, fontweight='bold')
plt.suptitle('Figure 1 (recreated): per-condition violation by SSD\n(thin = subjects, black = bootstrap mean + 95% CI)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 4. Figure 2a — violations by SSD, all conditions overlaid

The paper's Figure 2a plots one line per condition (no subject lines) plus a black group-mean line with 95% CI computed across conditions. Here we use one line per (Simple Stop + each memory load), and the black mean+CI bootstraps over **subjects pooled across conditions** at each SSD.


In [ ]:
# === Figure 2a: per-condition mean violation by SSD; group mean+CI overlay ===
fig, ax = plt.subplots(figsize=(11, 6.5))
all_subject_curves = []

for cond in CONDITIONS:
    if cond['load'] is None:
        per = race_simple.copy()
        ssd_col = 'trial_SSD'
    else:
        per = race_dual[race_dual['memory_trial_stimLength'] == cond['load']].copy()
        ssd_col = 'stop_trial_SSD'

    cond_mean_by_ssd = per.groupby(ssd_col)['mean_violation_ms'].mean()
    cond_n_by_ssd = per.groupby(ssd_col)['mean_violation_ms'].size()
    keep = cond_n_by_ssd >= 5
    cond_mean_by_ssd = cond_mean_by_ssd[keep]
    ax.plot(cond_mean_by_ssd.index, cond_mean_by_ssd.values,
            marker='o', color=cond['color'], linewidth=2.0, alpha=0.85,
            label=cond['label'])

    # Pool per-subject means at each SSD, tagging the SSD column uniformly.
    tmp = per[[ssd_col, 'mean_violation_ms']].rename(columns={ssd_col: 'ssd'})
    tmp['cond'] = cond['label']
    all_subject_curves.append(tmp)

pooled = pd.concat(all_subject_curves, ignore_index=True)
boot = bootstrap_mean_ci_by_ssd(pooled, ssd_col='ssd', value_col='mean_violation_ms')
if not boot.empty:
    ax.plot(boot['ssd'], boot['mean'], color='black', linewidth=4.0, zorder=10,
            label='Pooled mean (bootstrap 95% CI)')
    ax.fill_between(boot['ssd'], boot['lo'], boot['hi'],
                    color='black', alpha=0.18, zorder=9)

ax.axhline(0, color='black', linestyle=':', linewidth=1.2)
ax.set_xlabel('Stop-signal delay (ms)', fontsize=13, fontweight='bold')
ax.set_ylabel('Stop-failure RT − preceding go RT (ms)',
              fontsize=13, fontweight='bold')
ax.set_title('Figure 2a (recreated): violations by SSD across conditions',
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.legend(loc='best', fontsize=10, framealpha=0.9)
plt.tight_layout()
plt.show()


## 5. Figure 2b — proportion of *conditions* that violate at each SSD

For each SSD, count what fraction of conditions have a positive group-mean violation. In the paper, with 25 conditions, this curve is highly discriminating; here we have only ~4 conditions per experiment, so the curve will be coarser, but the shape is informative.


In [ ]:
# === Figure 2b: proportion of conditions that violate at each SSD ===
all_ssds = sorted(set(
    list(race_simple['trial_SSD'].dropna().unique())
    + list(race_dual['stop_trial_SSD'].dropna().unique())
))

cond_violates = {}  # ssd -> [bool per condition]
for cond in CONDITIONS:
    if cond['load'] is None:
        per = race_simple.copy()
        ssd_col = 'trial_SSD'
    else:
        per = race_dual[race_dual['memory_trial_stimLength'] == cond['load']].copy()
        ssd_col = 'stop_trial_SSD'
    grp = per.groupby(ssd_col)
    means = grp['mean_violation_ms'].mean()
    ns = grp['mean_violation_ms'].size()
    means = means[ns >= 5]
    for ssd in all_ssds:
        if ssd in means.index:
            cond_violates.setdefault(ssd, []).append(means.loc[ssd] > 0)

xs, ps, n_conds = [], [], []
for ssd in sorted(cond_violates):
    v = cond_violates[ssd]
    xs.append(ssd); ps.append(np.mean(v)); n_conds.append(len(v))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(xs, ps, marker='o', linewidth=3, color='black')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Half')
ax.set_xlabel('Stop-signal delay (ms)', fontsize=13, fontweight='bold')
ax.set_ylabel('P(conditions that violate)', fontsize=13, fontweight='bold')
ax.set_ylim(-0.02, 1.02)
ax.set_title('Figure 2b (recreated): proportion of conditions violating',
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.legend(loc='best', fontsize=10)
# Annotate n_conditions per SSD as a thin layer
for x, n in zip(xs, n_conds):
    ax.text(x, 1.04, f'n={n}', ha='center', va='bottom', fontsize=8, color='0.4')
plt.tight_layout()
plt.show()


## 6. Figure 2c — proportion of *subjects* who violate at each SSD

For each (condition, SSD), compute the fraction of subjects whose paired-preceding violation > 0. Plot one line per condition.


In [ ]:
# === Figure 2c: proportion of subjects who violate at each SSD ===
fig, ax = plt.subplots(figsize=(11, 6.5))
for cond in CONDITIONS:
    if cond['load'] is None:
        per = race_simple.copy()
        ssd_col = 'trial_SSD'
    else:
        per = race_dual[race_dual['memory_trial_stimLength'] == cond['load']].copy()
        ssd_col = 'stop_trial_SSD'
    grp = per.groupby(ssd_col)
    p_violate = grp['violator_at_ssd'].mean()
    ns = grp['violator_at_ssd'].size()
    keep = ns >= 5
    p_violate = p_violate[keep]
    ax.plot(p_violate.index, p_violate.values,
            marker='o', color=cond['color'], linewidth=2.5, label=cond['label'])

ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Half')
ax.set_xlabel('Stop-signal delay (ms)', fontsize=13, fontweight='bold')
ax.set_ylabel('P(subjects who violate)', fontsize=13, fontweight='bold')
ax.set_ylim(-0.02, 1.02)
ax.set_title('Figure 2c (recreated): proportion of subjects violating per SSD',
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.legend(loc='best', fontsize=10, framealpha=0.9)
plt.tight_layout()
plt.show()


## 7. Figure 2d — cumulative stop-trial distribution by SSD

The original Figure 2d is a task-design view: across conditions, what fraction of all stop trials have SSD ≤ x? With a tracking algorithm, this exposes how often subjects sit at short SSDs. Bissett uses it to argue that violation-prone short SSDs are common.

Two summary lines are plotted on top of the per-condition curves:
- **Simple Stop (red, solid)** — the reference cumulative distribution from the single-task block.
- **Mean across memory loads (black, dashed)** — the cumulative distribution averaged over the load conditions only, so the contrast with Simple Stop reflects the *cost of the memory task*, not a load × Simple-Stop average.

SSDs are read from the raw stop-trial data in `trial_wise_data_*` (before the n≥2-pair filter) so the picture reflects task design, not the diagnostic's filters.


In [ ]:
# === Figure 2d: cumulative stop-trial distribution by SSD ===
fig, ax = plt.subplots(figsize=(11, 6.5))

# Simple-stop SSDs from raw stop trials
simple_stop = trial_wise_data_stop[
    (trial_wise_data_stop['trial_SS_trial_type'] == 'stop')
]['trial_SSD'].dropna().values

# Dual-task: per memory load
load_ssds = []   # list of (label, color, vals) for loads only
for i, L in enumerate(LOADS):
    sub = trial_wise_data_wm[
        (trial_wise_data_wm['stop_trial_SS_trial_type'] == 'stop')
        & (trial_wise_data_wm['memory_trial_stimLength'] == L)
    ]['stop_trial_SSD'].dropna().values
    load_ssds.append((f'Load {int(L)}', LOAD_PALETTE[i % len(LOAD_PALETTE)], sub))

ssd_grid = np.arange(0, 1001, 50)

def _cum_curve(vals):
    return np.array([np.mean(vals <= x) for x in ssd_grid])

# Thin colored curves for each load condition
load_curves = []
for label, color, vals in load_ssds:
    if len(vals) == 0:
        continue
    cum = _cum_curve(vals)
    ax.plot(ssd_grid, cum, marker='o', linewidth=2.0, color=color,
            alpha=0.7, label=label)
    load_curves.append(cum)

# Simple Stop: highlighted reference curve (its own thick line)
if len(simple_stop) > 0:
    simple_cum = _cum_curve(simple_stop)
    ax.plot(ssd_grid, simple_cum, color=SIMPLE_COLOR, linewidth=4.5,
            marker='o', markersize=7, zorder=10,
            label=f'Simple Stop (reference)')

# Mean across loads only (NOT including Simple Stop) — black, dashed
if load_curves:
    mean_loads = np.mean(np.vstack(load_curves), axis=0)
    ax.plot(ssd_grid, mean_loads, color='black', linewidth=4.5,
            linestyle='--', zorder=11, label='Mean across memory loads')

ax.set_xlabel('Stop-signal delay (ms)', fontsize=13, fontweight='bold')
ax.set_ylabel('Cumulative P(stop trial at SSD ≤ x)', fontsize=13, fontweight='bold')
ax.set_ylim(-0.02, 1.02)
ax.set_title('Figure 2d (recreated): cumulative stop-trial distribution by SSD',
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.axvline(200, color='gray', linestyle=':', alpha=0.7,
           label='SSD = 200 ms (Bissett threshold)')
ax.legend(loc='best', fontsize=10, framealpha=0.9)
plt.tight_layout()
plt.show()


## 8. Figure 3a — SSRT (all SSDs) vs. SSRT (SSDs >= 200 ms)

For each condition, compute SSRT twice — once on all stop trials, once on SSD >= 200 ms — and plot the paired connecting lines. The original plots one line per condition (across the 25 conditions in the paper); here we plot one line per subject within each condition, and overlay the group mean.


In [ ]:
# === Figure 3a: SSRT all SSDs vs. SSDs >= 200 ms, paired lines per subject ===
ssrt_simple = ssrt_short_vs_long_comparison(
    trial_wise_data_stop, ssrt_fn=calculate_ssrt_integration,
    trial_type_col='trial_SS_trial_type', rt_col='trial_rt',
    correct_col='trial_correct_trial', ssd_col='trial_SSD',
    response_deadline_col='trial_trial_duration',
    subject_col='participant_id', ssd_threshold_ms=200.0,
)
ssrt_dual = ssrt_short_vs_long_comparison(
    trial_wise_data_wm, ssrt_fn=calculate_ssrt_integration,
    trial_type_col='stop_trial_SS_trial_type', rt_col='stop_trial_rt',
    correct_col='stop_trial_correct_trial', ssd_col='stop_trial_SSD',
    response_deadline_col='stop_trial_trial_duration',
    subject_col='participant_id', condition_col='memory_trial_stimLength',
    ssd_threshold_ms=200.0,
)

n_panels = len(CONDITIONS)
fig, axes = plt.subplots(1, n_panels, figsize=(4 * n_panels, 6.5), sharey=True)
if n_panels == 1:
    axes = [axes]
for ax, cond in zip(axes, CONDITIONS):
    if cond['load'] is None:
        sub = ssrt_simple
    else:
        sub = ssrt_dual[ssrt_dual['memory_trial_stimLength'] == cond['load']]
    sub = sub.dropna(subset=['ssrt_all_ssd', 'ssrt_long_ssd'])
    for _, row in sub.iterrows():
        ax.plot([0, 1], [row['ssrt_all_ssd'], row['ssrt_long_ssd']],
                color=cond['color'], alpha=0.45, linewidth=1.2)
    # Group means + 95% bootstrap CI on the difference at each x position
    means = [sub['ssrt_all_ssd'].mean(), sub['ssrt_long_ssd'].mean()]
    ax.plot([0, 1], means, color='black', linewidth=4, marker='D',
            markersize=10, zorder=10)
    # Annotate the per-condition mean diff
    diff = sub['ssrt_diff_ms'].mean()
    ax.text(0.5, ax.get_ylim()[1] if False else means[0] + 25,
            f'\nΔ = {diff:+.1f} ms\n(n={len(sub)})',
            ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['All SSDs', 'SSDs ≥ 200 ms'], fontsize=11)
    ax.set_xlim(-0.25, 1.25)
    ax.set_title(cond['label'], fontsize=13, fontweight='bold', color=cond['color'])
    ax.grid(True, alpha=0.3, axis='y')
axes[0].set_ylabel('SSRT (ms)', fontsize=13, fontweight='bold')
plt.suptitle('Figure 3a (recreated): SSRT all SSDs vs. SSDs ≥ 200 ms',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
